# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## My Lane as an ML Task

### Task Type: Ranking

This problem is best framed as a ranking task.

The goal is to rank content pages based on their potential need and opportunity for refresh. The output should help the content team decide which pages should be reviewed and updated first.

Ranking is more suitable than simple classification because the goal is not only to decide whether a page needs refresh or not, but to prioritize pages relative to each other.

## Target or Proxy

The ideal target would be the future improvement in content performance after a refresh. However, this outcome is not directly available in the dataset.

Therefore, I will create a proxy target using existing performance and freshness signals.

Potential signals for the refresh priority score include:

- Low CTR
- Low engagement rate
- Older content (`days_since_last_update`)
- Declining trends (`trend_direction`, `trend_pct`)
- High impressions but lower clicks

The proxy target represents an estimate of which pages may have higher refresh opportunities, not a guaranteed outcome.

## Success Metric

Since this is a ranking problem, ranking-based metrics are appropriate.

The main evaluation metrics would be:

- Precision@K: Measures how many of the top recommended pages are actually valuable refresh candidates.
- NDCG: Measures whether the highest-priority pages appear near the top of the ranking list.

A successful system would help the content team find valuable pages earlier and reduce manual effort.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [5]:
!git clone https://github.com/daniishkumar/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 197, done.
remote: Counting objects: 100% (197/197), done.
remote: Compressing objects: 100% (109/109), done.
remote: Total 197 (delta 85), reused 169 (delta 68), pack-reused 0 (from 0)
Receiving objects: 100% (197/197), 1.91 MiB | 13.43 MiB/s, done.
Resolving deltas: 100% (85/85), done.


In [6]:
import pandas as pd

df = pd.read_csv("flyrank-ml-internship/data/raw/content_refresh_anonymized.csv")

df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [7]:
df.shape

(30000, 44)

In [9]:
df[
[
"content_id",
"content_age_days",
"days_since_last_update",
"impressions_90d",
"clicks_90d",
"ctr",
"engagement_rate"
]
].head()

,content_id,content_age_days,days_since_last_update,impressions_90d,clicks_90d,ctr,engagement_rate
0,content_304f48230142,187,20,3803,29,0.76,5.88
1,content_a1fb4e703a9e,445,25,15320,7,0.05,0.00
2,content_9aa793d4d895,141,20,12581,11,0.09,0.00
3,content_331d6c4de07b,463,22,11751,58,0.49,1.28
4,content_d99b7a2d90ca,263,14,19140,24,0.13,0.00


## Unit of Analysis

The unit of analysis is:

**One row = one content page**

Each row represents an individual piece of content that can be evaluated for refresh priority.

The ML system would generate recommendations at the content/page level.

In [11]:
df["refresh_priority_score"] = (
    (df["days_since_last_update"] / df["days_since_last_update"].max()) +
    (1 - df["ctr"]) +
    (1 - df["engagement_rate"])
)

df[["content_id", "refresh_priority_score"]].head()

,content_id,refresh_priority_score
0,content_304f48230142,-4.586381
1,content_a1fb4e703a9e,2.017024
2,content_9aa793d4d895,1.963619
3,content_331d6c4de07b,0.288981
4,content_d99b7a2d90ca,1.907534


The `refresh_priority_score` is a proxy target showing how a future ML system could rank pages.

A higher score represents a page that may deserve more attention based on available signals.

## 5. Why ML beats a fixed rule here

## Why ML Beats a Fixed Rule

A fixed rule could be:

"Refresh every page older than 365 days."

However, content performance depends on many factors. An old page may still perform well, while a newer page may need improvement.

Machine learning can combine multiple signals such as freshness, traffic, engagement, search performance, and trends to create a more flexible ranking.

This allows the team to prioritize pages using evidence instead of a single manually chosen rule.

## Self-Check

- Defined the ML task type as ranking.  
- Identified a target/proxy.  
- Selected suitable success metrics.  
- Showed the unit of analysis using a real dataframe.  
- Explained why ML is better than a fixed rule.  
- Connected the output to a real content action.